<a href="https://colab.research.google.com/github/amit-sw/colab_notebooks/blob/main/2026_RAG_chiller.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install openai --quiet
import os
from google.colab import userdata

os.environ['OPENAI_API_KEY']=userdata.get('OPENAI_API_KEY')

# Classification with raw OpenAI calls

In [2]:
from openai import OpenAI
import json
import textwrap

system_command="Is this sentiment positive or negative?"
user_input="I am feeling awful."

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
response = client.responses.create(model="gpt-5.4-nano",input=user_input,instructions=system_command,reasoning={"effort": "low"})

classification = response.output_text

print(textwrap.fill(f"{classification} \nInput was: {user_input}",width=100,replace_whitespace=False))

Negative — “I am feeling awful” expresses a clearly negative emotion. 
Input was: I am feeling
awful.


# Question-Answer with raw OpenAI calls

In [3]:
from openai import OpenAI
import json

system_command="Please answer this question from the user. Keep the answer short and concise."

def answer_question(user_input):
  client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
  response = client.responses.create(model="gpt-5.6-luna",input=user_input,instructions=system_command,reasoning={"effort": "low"})

  msg = response.output_text
  return msg

In [4]:
answer_question("What is the capital of France?")

'The capital of France is Paris.'

In [5]:
import textwrap
answer=answer_question("What did Trane announce in March 2026?")
print(textwrap.fill(answer,width=100))

March 2026 is in the future, so Trane has not made any verifiable announcements for that month yet.


# Augmented Question-Answer

In [6]:
def augmentation_data():
  new_information="""
  Trane Technologies Optimizes Industry‑First Thermal Management Reference Design for AI Factories, Introduces Two New Designs
Mar 16, 2026
Company improves design efficiency by nearly 10% and launches two additional Trane® Continuum Rubin DSX reference designs

SWORDS, Ireland--(BUSINESS WIRE)-- Trane Technologies (NYSE:TT), a global climate innovator, today announced major enhancements to its industry-first comprehensive thermal management reference design for gigawatt-scale AI factories and unveiled two Trane Continuum Rubin DSX reference designs.

Engineered specifically to integrate with the NVIDIA® Omniverse™ DSX Blueprint for AI data centers, the new system optimizations achieve a nearly 10% improvement in overall thermal management performance compared to the original 1-gigawatt reference design announced in October. This frees up 22 megawatts of cooling capacity that can be redirected to IT power, helping boost compute output without increasing total energy consumption.

“Since the launch of our industry-first thermal management system reference design, our team has continued to work hand-in-hand with NVIDIA to push the boundaries of efficiency, scalability and intelligent simulation for gigawatt-scale AI infrastructure,” said Mauro Atalla, Senior Vice President and Chief Technology and Sustainability Officer, Trane Technologies. “These latest advancements represent a major step forward in helping enable the world’s most demanding AI and high-performance computing environments to scale sustainably, reliably and with accelerated digital insight.”

“Efficiently scaling gigawatt-scale AI factories requires a fundamental shift in how we approach thermal management and data center infrastructure simulation,” said Vladimir Troy, Vice President of AI Infrastructure at NVIDIA. “Trane Technologies’ integration with the NVIDIA Omniverse DSX blueprint enables the creation of high-fidelity digital twins that help operators optimize cooling performance and maximize energy efficiency for the next generation of AI workloads.”

Through continued collaboration with NVIDIA, including as an NVIDIA Partner Network member, Trane Technologies has also expanded its Trane Continuum Rubin DSX reference design portfolio with two additional high-efficiency solutions for large-scale AI deployments:

250-Megawatt Duplex Simplified System Design: This new design, available now, supports extended free cooling use and incorporates integrated heat recovery, which helps reduce system complexity and results in a 14% increase in thermal management system efficiency with 10% of the heat rejection load going to heat recovery.
1-Gigawatt AI Factory Mag-Bearing Air-Cooled System Architecture: This new design, available soon, features a streamlined air-cooled approach using 3-megawatt units to help reduce equipment count and eliminate the need for integrated waterside economizers. The architecture incorporates Trane’s latest magnetic-bearing air-cooled chiller, providing oil-free operation, high efficiency and quieter, more efficient performance through Trane chiller plant controls.
Trane Technologies has also advanced its digital capabilities for adopting the Omniverse DSX Blueprint with more automated, scalable OpenUSD based SimReady assets. Enhanced with structured metadata, these assets will support upcoming reference design updates, helping improve configurability, accuracy and readiness for high‑scale digital‑twin and AI‑driven simulation workflows.

About Trane Technologies

Trane Technologies is a global climate innovator. Through our strategic brands Trane® and Thermo King®, and our portfolio of environmentally responsible products and services, we bring efficient and sustainable climate solutions to buildings, homes and transportation. For more on Trane Technologies, visit www.tranetechnologies.com.

About Trane

Trane® – by Trane Technologies (NYSE: TT), a global climate innovator – creates comfortable, energy efficient indoor environments through a broad portfolio of heating, ventilating and air conditioning systems and controls, services, parts and supply. For more information, please visit www.trane.com or www.tranetechnologies.com.

"""
  return new_information

In [7]:
from openai import OpenAI
import json


system_command="Please answer this question from the user. Keep the answer short and concise."

def answer_question_augmented(user_input):
  client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
  response = client.responses.create(model="gpt-5.4-nano",input=
                                     [{'role':'user',"content":user_input},
                                      {'role':'assistant',"content":augmentation_data()}],
                                     instructions=system_command,reasoning={"effort": "low"})

  msg = response.output_text
  return msg

In [8]:
import textwrap
answer=answer_question_augmented("What did Trane announce in march 2026?")
print(textwrap.fill(answer,width=100))

In March 2026, Trane announced enhancements to its industry-first thermal management reference
design for gigawatt-scale AI factories—improving overall thermal management performance by nearly
10%—and it also introduced two new Trane Continuum Rubin DSX reference designs (a 250‑MW duplex
simplified system and a 1‑GW AI factory mag-bearing air-cooled system).


# Retrieval-Augmented Generation: Storage

---



In [9]:
!pip install langchain faiss-cpu transformers --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 38.8 MB/s eta 0:00:00


In [10]:
import faiss
import numpy as np
from openai import OpenAI

client=OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

def embed_text_openai(text):
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )
    embeddings = response.data[0].embedding
    return np.array(embeddings)

def chunk_text(text, chunk_size=1000, overlap=100):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)
    return chunks

def store_in_faiss(embeddings):
    dimension = embeddings.shape[1]
    index = faiss.IndexFlatL2(dimension)
    index.add(embeddings)
    return index

In [12]:
import torch
with open('/content/trane_chiller_CDIG-TUY11FE-EN_07112026.md', 'r',  encoding='latin-1') as file:
    text = file.read()
chunks = chunk_text(text)
embeddings = np.vstack([embed_text_openai(chunk) for chunk in chunks])
index = store_in_faiss(embeddings)
faiss.write_index(index, "faiss_index.index")

print(f"Chunk count: {len(chunks)}")
for i in range(3):
  print(chunks[i]+"\n***********\n")

Chunk count: 50
# CenTraVacÂ® Water-cooled Chillers â Models CDHF and CDHG **Installation, Operation, and Maintenance â With Symbioâ¢ Controls** **Document:** CDIG-TUY11FE-EN **Date:** April 2026 > Markdown transcription of the supplied 104-page Trane manual. The PDF text layer is preserved page-by-page. Figures/diagrams are represented by their captions, labels, legends, callouts, and concise visual descriptions where the visual conveys information not adequately represented by running text. Page headers/footers may remain where they carry document context. --- ## PDF Page 1 Installation, Operation, and Maintenance CenTraVacÂ® Water-cooled Chillers Models CDHF and CDHG With Symbioâ¢Controls Models: CDHF, CDHG X39641428005 SAFETY WARNING Only qualified personnel should install and service the equipment. The installation, starting up, and servicing of heating, ventilating, and air-conditioning equipment can be hazardous and requires specific knowledge and training. Improperly inst

In [13]:
def retrieve_similar_chunks(query, index, chunks, top_k=3):
    query_embedding = embed_text_openai(query).reshape(1,-1)
    distances, indices = index.search(query_embedding, top_k)
    return [(chunks[idx], distances[0][i]) for i, idx in enumerate(indices[0])]

#query = input("Enter a query:")
query = "What precautions should I take while working with the unit?"
print(f"\n\nQuery: {query}")
top_k = 3  # Number of similar chunks to retrieve
similar_chunks = retrieve_similar_chunks(query, index, chunks, top_k)

for chunk, distance in similar_chunks:
  print(f"Distance: {distance} Chunk: {chunk}\n")



Query: What precautions should I take while working with the unit?
Distance: 1.049742341041565 Chunk: # CenTraVacÂ® Water-cooled Chillers â Models CDHF and CDHG **Installation, Operation, and Maintenance â With Symbioâ¢ Controls** **Document:** CDIG-TUY11FE-EN **Date:** April 2026 > Markdown transcription of the supplied 104-page Trane manual. The PDF text layer is preserved page-by-page. Figures/diagrams are represented by their captions, labels, legends, callouts, and concise visual descriptions where the visual conveys information not adequately represented by running text. Page headers/footers may remain where they carry document context. --- ## PDF Page 1 Installation, Operation, and Maintenance CenTraVacÂ® Water-cooled Chillers Models CDHF and CDHG With Symbioâ¢Controls Models: CDHF, CDHG X39641428005 SAFETY WARNING Only qualified personnel should install and service the equipment. The installation, starting up, and servicing of heating, ventilating, and air-conditioning 

In [14]:
def customized_augmentation_data(input_question : str) -> str:
  #
  # Return information based on the input question asked.
  #
  query_embedding = embed_text_openai(query).reshape(1,-1)
  distances, indices = index.search(query_embedding, top_k)
  new_information="\n\n".join(chunks[idx] for i, idx in enumerate(indices[0]))
  print(f"Retrieval:\n\\n {new_information}\nn")
  return new_information

# RAG: With retrieval

In [15]:
from openai import OpenAI
import json


system_command="Please answer this question from the user. Keep the answer short and concise. Only use the provided information to answer the question."

def answer_question_augmented(user_input):
  client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
  response = client.responses.create(model="gpt-5.6-luna",input=
                                     [{'role':'user',"content":user_input},
                                      {'role':'assistant',"content":customized_augmentation_data(user_input)}],
                                     instructions=system_command,reasoning={"effort": "low"})

  msg = response.output_text
  return msg

In [16]:
import textwrap
answer=answer_question_augmented("What precautions should I take while working with the unit?")
print("\n*** ANSWER ***\n"+textwrap.fill(answer,width=100))

Retrieval:
\n # CenTraVacÂ® Water-cooled Chillers â Models CDHF and CDHG **Installation, Operation, and Maintenance â With Symbioâ¢ Controls** **Document:** CDIG-TUY11FE-EN **Date:** April 2026 > Markdown transcription of the supplied 104-page Trane manual. The PDF text layer is preserved page-by-page. Figures/diagrams are represented by their captions, labels, legends, callouts, and concise visual descriptions where the visual conveys information not adequately represented by running text. Page headers/footers may remain where they carry document context. --- ## PDF Page 1 Installation, Operation, and Maintenance CenTraVacÂ® Water-cooled Chillers Models CDHF and CDHG With Symbioâ¢Controls Models: CDHF, CDHG X39641428005 SAFETY WARNING Only qualified personnel should install and service the equipment. The installation, starting up, and servicing of heating, ventilating, and air-conditioning equipment can be hazardous and requires specific knowledge and training. Improperly instal

# Extract data from PDF file

In [17]:
!pip install -q pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 385.1/385.1 kB 4.0 MB/s eta 0:00:00


In [18]:
from pypdf import PdfReader

file_path = "/content/trane_chiller_CDHF-SVX004E-EN_04112026.pdf"

reader = PdfReader(file_path, strict=False)

text_content = []

for page in reader.pages:
    text = page.extract_text()
    if text:
        text_content.append(text)

full_text = "\n\n".join(text_content)

print(f"Pages: {len(reader.pages)}")
print(f"Characters extracted: {len(full_text):,}")
print(full_text[:2000])

Pages: 104
Characters extracted: 233,704
X39641428005
SAFETY WARNING
Onlyqualifiedpersonnelshouldinstallandservicetheequipment.Theinstallation,startingup,andservicingofheating,ventilating,andair-conditioningequipment
canbehazardousandrequiresspecificknowledgeandtraining.Improperlyinstalled,adjustedoralteredequipmentbyanunqualifiedpersoncouldresultindeathor
seriousinjury.Whenworkingontheequipment,observeallprecautionsintheliteratureandonthetags,stickers,andlabelsthatareattachedtotheequipment.
April2026 CDHF-SVX004E-EN
Installation,Operation,andMaintenance
CenTraVac® Water-cooled Chillers
Models CDHF and CDHG
WithSymbio™Controls
Models:CDHF,CDHG

©2026Trane CDHF-SVX004E-EN
Introduction
Readthismanualthoroughlybeforeoperatingorservicing
thisunit.
Warnings, Cautions, and Notices
Safetyadvisoriesappearthroughoutthismanualas
required.Yourpersonalsafetyandtheproperoperationof
thismachinedependuponthestrictobservanceofthese
precautions.
Thethreetypesofadvisoriesaredefinedasfollows:
WARNING
Ind